In [2]:
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
import pandas as pd
from langchain_chroma import Chroma
import torch
# print(torch.__version__, torch.version.cuda)
# print(torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [3]:
#importing dataset



# 1. Initialize the loader with your PDF file
loader = PyPDFLoader("S:/Projects/rag-dissertation/datasets/Indian Legal Documents LuRA/legaldoc.pdf")

# 2. Load and split the PDF into a list of Documents (one per page)
docs = loader.load()

# Or load and split into chunks directly
pages = loader.load_and_split()

print(f"Total pages loaded: {len(docs)}")
print(docs[0].page_content)  # Content of the first page
print(docs[0].metadata)  # Page number and source path

Total pages loaded: 103
What is Lease Deed (for a term of years) Rent Agreement? 
A rent or lease agreement is an agreement that lays down pre-discussed terms and conditions 
under which a property is to be rented or leased between a tenant and landlord. A lease 
agreement is essentially an agreement for leasing of an immovable property for up to 11 
months. A rent agreement can be an agreement of more than a year. All important clauses 
stating the terms, conditions and promises between both the parties must be included in a 
lease or rent agreement.  
Why is Lease Deed (for a term of years) Rent Agreement required? 
A lease or rent agreement is an essential document that evidences the leasing or renting of an 
immovable property. All terms relating to such lease/rent including the time period for such 
lease and the purpose for which the property must be used, among other terms, must form a 
part of a rent agreement. Such an agreement serves the purpose of a reference document for al

In [ ]:
#loading dataset in langchain

# docs = []
# for col in df.columns:
#     if col == "Unnamed: 0":
#         continue
#     text = "\n".join(df[col].dropna().astype(str))
#     docs.append(Document(page_content=text, metadata={"column": col}))

# print(len(docs))   # = number of columns
# print(docs[0].metadata)
# print(type(docs[0].page_content))

1
{'column': 'deep_learning'}
<class 'str'>


In [4]:
#Spliting text

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(docs)
print(len(texts)) 
print(texts[5].page_content)

301
rent to be paid on or before the ___ day of every succeeding month regularly. 
 
2.     The Lessee hereby for himself, his heirs, executors, administrators and assigns and to the 
intent that the obligations herein contained shall continue throughout the term hereby created 
covenants with the Lessor as follows: 
 
a.     To pay the ground rent hereby reserved on the days and in the manner aforesaid clear of all 
deductions. The first of such monthly rent as hereinbefore provided shall be paid on the.... of 
....... and the subsequent rent shall be paid on the.... day of every succeeding month regularly 
and If the-ground rent is not paid on the due dates the Lessee shall pay interest thereon at the 
rate of ....... % per annum from the due date till payment, though the payment of Interest shall 
not entitle the Lessee to make default in payment of rent on due dates. 
 
b.    To bear pay and discharge the existing and future rates. taxes and assessment duties, cess,


In [4]:
#Embeddings Model 

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"batch_size": 512},
    )

embeddings._client.half()                 # fp16
embeddings._client.max_seq_length = 256   # truncate long abstracts

query_embedding = embeddings.embed_query("What is a sentence embedding?")
doc_embeddings = embeddings.embed_documents(
    [
        "Sentence embeddings map text to dense vectors.",
        "LangChain provides a standard Embeddings interface.",
    ]
)

print(type(embeddings))
print(hasattr(embeddings, "_client"), hasattr(embeddings, "client"))

s:\Projects\rag-dissertation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 759.20it/s]


<class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>
True False


In [7]:
m = embeddings._client
print(m.max_seq_length)                  # 256
print(next(m.parameters()).dtype)        # torch.float16
print(next(m.parameters()).device)       # cuda:0

256
torch.float16
cuda:0


In [5]:
# Embeddings & vector storage
#
# This cell only READS the Chroma DB that was persisted to ./chroma_langchain_db
# (src/chroma_langchain_db) -- it does not re-embed the dataset on every run.

from pathlib import Path

CHROMA_DIR = Path("chroma_langchain_db")
if not CHROMA_DIR.exists():
    raise FileNotFoundError(
        f"Local Chroma DB not found at {CHROMA_DIR.resolve()}. "
        "Run the ingestion cells once with REBUILD_INDEX = True to create it."
    )

vector_store = Chroma(
    collection_name="Legal_Embeddings",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

# Default LangChain retriever over the persisted collection.
retriever = vector_store.as_retriever()

print(f"Loaded {vector_store._collection.count()} chunks from {CHROMA_DIR.resolve()}")


Loaded 301 chunks from S:\Projects\rag-dissertation\src\chroma_langchain_db


In [9]:
# Timeline of storage
#
# The index already exists on disk, so ingestion is opt-in: flip REBUILD_INDEX to
# True only when you want to (re)embed the split texts into the collection.

from tqdm import tqdm

REBUILD_INDEX = True

if REBUILD_INDEX:
    max_bs = vector_store._client.get_max_batch_size()   # 5461 for you
    BATCH = min(2000, max_bs)                            # stay safely under it

    for i in tqdm(range(0, len(texts), BATCH)):
        batch = texts[i:i + BATCH]
        ids = [str(d.metadata.get("pmid", i + j)) for j, d in enumerate(batch)]
        vector_store.add_documents(batch, ids=ids)
else:
    print("Skipping ingestion; reading the existing local Chroma DB as-is.")


100%|██████████| 1/1 [00:39<00:00, 39.85s/it]


In [12]:
# Retrieve evidence for the same question that will be evaluated.
query = "What should a Legal Notice for Recovery of Money cover?"
retrieved_docs = retriever.invoke(query)
retrieval_context = [doc.page_content for doc in retrieved_docs]
if not retrieval_context:
    raise ValueError("The retriever returned no context; cannot evaluate a RAG answer.")
print(f"Retrieved {len(retrieval_context)} context chunks.")

# Show the retrieved chunks (rank, metadata, text) so the evidence is visible.
for rank, doc in enumerate(retrieved_docs, start=1):
    meta = ", ".join(f"{k}={v}" for k, v in doc.metadata.items()) or "none"
    print(f"--- chunk {rank}/{len(retrieved_docs)} [{meta}] ---")
    print(doc.page_content)
    print()


Retrieved 4 context chunks.
--- chunk 1/4 [creator=Microsoft® Word for Microsoft 365, page=47, creationdate=2025-01-29T23:19:52+05:30, page_label=48, author=keyush nisar, total_pages=103, moddate=2025-01-29T23:19:52+05:30, producer=Microsoft® Word for Microsoft 365, source=S:/Projects/rag-dissertation/datasets/Indian Legal Documents LuRA/legaldoc.pdf] ---
One may send a notice for recovery of money from your employer, friend or any other party who 
owes you money. The first step in recovery should be to send a legal notice since most recovery 
cases get resolved in this step itself and you do not need to proceed to Court. One must send a 
well-drafted and legally sound notice to ensure that you get a prompt response on your notice 
and you have maximum opportunity to recover your money. 
What should a Legal Notice for Recovery of Money cover? 
The legal notice for recovery of money must contain 
o The legal notice must be drafted in the letterhead of an advocate which is to be 
specifi

In [7]:
#Setting up LLM

llm = ChatOllama(
    model="qwen3.5:2b", 
    temperature = 0.5,
    reasoning=False,
    
)

In [13]:
#Pre Retrieval - Baseline
# context_text = "\n\n".join(retrieval_context)

response = llm.invoke(query)
print(response.content)


A **Legal Notice** is a formal written communication sent by one party (the Creditor) to another (the Debtor) demanding payment or performance of an obligation. It serves as a precursor to legal action, giving the debtor a final opportunity to resolve the dispute amicably before proceeding to court.

To make your notice effective and legally sound, it should comprehensively cover the following key elements:

### 1. Identification of Parties
Clearly state who is sending the notice and who is receiving it. This establishes jurisdiction and authority.
*   **Sender Details**: Full name, address, contact information, and representation details (e.g., "Lawyer for [Company Name]" or "Individual").
*   **Receiver Details**: Full name, address, and contact information of the Debtor.

### 2. Statement of the Debt
This is the core section where you detail the financial obligation. It must be specific to avoid ambiguity.
*   **Amount Owed**: Clearly state the principal amount due.
*   **Interest/C

In [14]:
#Pre Retrieval - Baseline Eval

import os

# The local judge can be slow, so allow longer local judge calls.
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"

actual_context =[
    """The legal notice for recovery of money must contain

>The legal notice must be drafted in the letterhead of an advocate which is to be specific and proper.

>It should contain addresses and contact details of the advocate.

>The date on which the legal notice is issued including the name, address, and contact details of the person to whom the legal notice is issued is to be stated.

>Since the legal notice for recovery of money is going from the client's end, the name and details of the client should be mentioned.

>It should be made clear in the notice as to how your right has been infringed due to the act or omission by the opposite party and for that what you want from him. A specific direction must be given to the opposite party along with a time limit.

>The notice must be signed (with date) by the advocate and the sender."""
]

from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    HallucinationMetric,
)
from deepeval.models.llms.ollama_model import OllamaModel, retry_ollama
from deepeval.test_case import Audio, LLMTestCase
from pydantic import BaseModel
from typing import List, Optional, Tuple, Union


class NonThinkingOllamaModel(OllamaModel):
    @retry_ollama
    def generate(
        self,
        prompt: str,
        schema: Optional[BaseModel] = None,
        audio: Optional[List[Audio]] = None,
    ) -> Tuple[Union[str, BaseModel], float]:
        response = self.load_model().chat(
            model=self.name,
            messages=self._prompt_messages(prompt, audio),
            think=False,
            format=schema.model_json_schema() if schema else None,
            options={
                "temperature": self.temperature,
                **self.generation_kwargs,
            },
        )
        content = response.message.content
        if not content:
            raise ValueError(
                "Ollama returned empty content for the DeepEval judge response."
            )
        result = schema.model_validate_json(content) if schema else content
        return result, 0

# Use a separate local judge; Qwen 3.5 otherwise returns reasoning instead of JSON.
evaluation_model = NonThinkingOllamaModel(
    model="gemma4:e4b",
    base_url="http://localhost:11434",
    temperature=0,
    # reasoning=False,
)

test_case = LLMTestCase(
    input=query,
    actual_output=response.content,
    context=actual_context,
    retrieval_context=retrieval_context,
)

metrics = {
    "hallucination": HallucinationMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "faithfulness": FaithfulnessMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "answer relevancy": AnswerRelevancyMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
}

for name, metric in metrics.items():
    metric.measure(test_case)
    print(f"{name}: {metric.score:.3f}")
    print(f"  passed: {metric.is_successful()}")
    print(f"  reason: {metric.reason}")


s:\Projects\rag-dissertation\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

hallucination: 0.000
  passed: False
  reason: The score is 0.00 because the actual output omits specific, required details from the context, such as the notice needing to be on an advocate's letterhead, including contact details, and the explicit mention of the client's name and details.


faithfulness: 1.000
  passed: True
  reason: The score is 1.00 because there are no contradictions, meaning the actual output is perfectly faithful to the retrieval context.


answer relevancy: 1.000
  passed: True
  reason: The score is 1.00 because the output appears to be entirely relevant to the input, and there were no irrelevant statements noted.


In [15]:
# Post Retrieval Query
context_text = "\n\n".join(retrieval_context)
prompt = f"""Answer the question using only the supplied context. If the context does not contain the answer, say so.

Context:
{context_text}

Question: {query}
Answer:"""
response = llm.invoke(prompt)
print(response.content)


A legal notice for recovery of money must contain the following:

*   The legal notice must be drafted in the letterhead of an advocate which is to be specific and proper.
*   It should contain addresses and contact details of the advocate.
*   The date on which the legal notice is issued, including the name, address, and contact details of the person to whom the legal notice is issued.
*   Since it goes from the client's end, the notice serves to give a reasonable time for payment and acts as valid proof before a Court that an amount is owed.

Additionally, the notice should:
*   Mention the name and details of the client.
*   Clearly state how the right has been infringed by the opposite party and what is desired.
*   Include a specific direction along with a time limit.
*   Be signed (with date) by the advocate and the sender.


In [16]:
#Post Retrieval Eval
import os

# The local judge can be slow, so allow longer local judge calls.
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"

actual_context =[
    """The legal notice for recovery of money must contain
    
    >The legal notice must be drafted in the letterhead of an advocate which is to be specific and proper.
    
    >It should contain addresses and contact details of the advocate.
    
    >The date on which the legal notice is issued including the name, address, and contact details of the person to whom the legal notice is issued is to be stated.
    
    >Since the legal notice for recovery of money is going from the client’s end, the name and details of the client should be mentioned.
    
    >It should be made clear in the notice as to how your right has been infringed due to the act or omission by the opposite party and for that what you want from him. A specific direction must be given to the opposite party along with a time limit.
    
    >The notice must be signed (with date) by the advocate and the sender."""
]

from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    HallucinationMetric,
)
from deepeval.models.llms.ollama_model import OllamaModel, retry_ollama
from deepeval.test_case import Audio, LLMTestCase
from pydantic import BaseModel
from typing import List, Optional, Tuple, Union


class NonThinkingOllamaModel(OllamaModel):
    @retry_ollama
    def generate(
        self,
        prompt: str,
        schema: Optional[BaseModel] = None,
        audio: Optional[List[Audio]] = None,
    ) -> Tuple[Union[str, BaseModel], float]:
        response = self.load_model().chat(
            model=self.name,
            messages=self._prompt_messages(prompt, audio),
            think=False,
            format=schema.model_json_schema() if schema else None,
            options={
                "temperature": self.temperature,
                **self.generation_kwargs,
            },
        )
        content = response.message.content
        if not content:
            raise ValueError(
                "Ollama returned empty content for the DeepEval judge response."
            )
        result = schema.model_validate_json(content) if schema else content
        return result, 0

# Use a separate local judge; Qwen 3.5 otherwise returns reasoning instead of JSON.
evaluation_model = NonThinkingOllamaModel(
    model="gemma4:e4b",
    base_url="http://localhost:11434",
    temperature=0,
)

test_case = LLMTestCase(
    input=query,
    actual_output=response.content,
    context=actual_context,
    retrieval_context=retrieval_context,
)

metrics = {
    "hallucination": HallucinationMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "faithfulness": FaithfulnessMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "answer relevancy": AnswerRelevancyMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
}

for name, metric in metrics.items():
    metric.measure(test_case)
    print(f"{name}: {metric.score:.3f}")
    print(f"  passed: {metric.is_successful()}")
    print(f"  reason: {metric.reason}")


s:\Projects\rag-dissertation\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

hallucination: 1.000
  passed: True
  reason: The score is 1.00 because the actual output accurately reflects all the points mentioned in the context regarding the contents of a legal notice for recovery of money, including the drafting on advocate's letterhead, inclusion of addresses/contact details, date and recipient details, client details, clarity on infringement and desired remedy with a time limit, and signatures.


faithfulness: 1.000
  passed: True
  reason: The score is 1.00 because there are no contradictions, meaning the actual output is perfectly faithful to the retrieval context.


answer relevancy: 1.000
  passed: True
  reason: The score is 1.00 because the output appears to be entirely relevant to the input, and there were no irrelevant statements noted.
